In [1]:
import warnings
warnings.filterwarnings("ignore", message="`langchain-community` is being sunset")
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import WikipediaLoader, Docx2txtLoader, PyPDFLoader, TextLoader

In [2]:
from langchain_chroma import Chroma

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os

In [4]:
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)

In [6]:
## embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
embeddings_model = OpenAIEmbeddings()

In [8]:
vector_db = Chroma("tourist_info", embeddings_model)

In [9]:
def split_and_import(loader):
    chunks = text_splitter.split_documents(loader.load())
    vector_db.add_documents(chunks)
    print(f"Ingested chunks created by {loader}")

In [10]:
import wikipedia
# Descriptive UA avoids Wikimedia's 429 on the lib's shared default UA
wikipedia.wikipedia.USER_AGENT = "tourist-info-notebook/1.0 (mic.a.elle.chlon@gmail.com)"

# wikipedia_loader = WikipediaLoader(query="Paestum")
# split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7fe7033a2120>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x7fe7033a3770>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x7fe6fe96b4d0>


## 7.2.3/ Ingest multiple documents from a folder

### Ingesting the files from the folder 

### Alternative: Ingestive all files with DirectoryLoader

In [11]:
from langchain_community.document_loaders import DirectoryLoader

folder_path = "CilentoTouristInfo"
directory_loader = DirectoryLoader(
    folder_path,
    glob=["**/*.docx", "**/*.pdf", "**/*.txt"],
    show_progress=True,
)
split_and_import(directory_loader)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:12<00:00,  1.31it/s]


Ingested chunks created by <langchain_community.document_loaders.directory.DirectoryLoader object at 0x7fe6fe96b620>


## 7.3.1/ Querying the vector store directly

In [12]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query=query, k=4)
print(results)

[Document(id='f1a8369d-4bda-4c39-8745-a410e2eb8ef1', metadata={'source': 'Paestum/Paestum-Britannica.docx'}, page_content='Poseidonia was probably founded about 600\xa0BC\xa0by Greek colonists from\xa0Sybaris, along the\xa0Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city came under the domination of the\xa0Lucanians\xa0(an\xa0indigenous\xa0Italic people) sometime before 400\xa0BC, after which its name was changed to Paestum. Alexander, the king of Epirus, defeated the Lucanians at Paestum about 332\xa0BC, but the city remained Lucanian until 273, when it came under'), Document(id='89d2e858-bca8-4581-af16-73f4d3dffbb0', metadata={'total_pages': 4, 'page_label': '2', 'page': 1, 'creationdate': 'D:20231028153815', 'creator': 'PDFium', 'producer': 'PDFium', 'source': 'Paestum/PaestumRevisited.pdf'}, page_content='Southern Italy at the time: Lucanization.63 But the evidence from Poseidonia suggests that the Greek p

## 7.3.2/ Asking a question through a LangChain chain

In [13]:
from langchain_core.prompts import PromptTemplate

In [14]:
rag_prompt_template = """Use the following pieces of context
to answer the question at the end.
If you don't know the answer, just say that you don't know,
don't try to make up an answer.
Use three sentences maximum and keep the answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

In [15]:
retriever = vector_db.as_retriever()

In [16]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

from langchain_openai import ChatOpenAI

chatbot = ChatOpenAI(api_key=OPENAI_API_KEY, model='gpt-5-nano')

In [17]:
rag_chain = {"context": retriever, 
            "question": question_feeder}|rag_prompt|chatbot

In [18]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [19]:
question = """Where was Poseidonia and who renamed
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(answer.content)

- Poseidonia was located in southern Italy along the Gulf of Taranto. 
- It was renamed Paestum by the Lucanians sometime before 400 BC. 
- Source: Paestum-Britannica.docx (the Poseidonia/Paestum entry).
